In [10]:
import pandas as pd
import numpy as np
import tensorflow as tf
#import tensorflow_addons as tfa
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.layers import LSTM,Bidirectional,GRU
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.utils import to_categorical
import datetime
import io
import itertools
# import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from sklearn.model_selection import KFold
from sklearn.metrics import classification_report

import sys
import os
# Obtener la ruta del directorio actual
os.chdir('/home/rgadea/experimentos_software_2024/nuevas_investigaciones_alimentos_2024')
current_dir = os.getcwd()
print(current_dir)

# Construir la ruta relativa al directorio que quieres agregar
relative_dir = os.path.join(current_dir, 'mis_pkgs/')

# Agregar la ruta relativa al sys.path
sys.path.insert(0, relative_dir)

#from MIOPATIA_db import DB_management as db 


/home/rgadea/experimentos_software_2024/nuevas_investigaciones_alimentos_2024


In [11]:
numero_muestras=201
numero_clases=2
entrada=[4,8]
numero_entradas =2
numero_epochs=20000

Voy a quedarme con los 50 atunes P1 para obtener conjunto de training y validacion

In [12]:
filename = "COPIA_PANDAS/medidas_agilent_2023_y_2024_201_puntos_clasificados.hdf"
with pd.HDFStore(filename,complib="zlib",complevel=4) as hdf_db:
    pre_p_e1  = hdf_db.get('data/pollos_estado')
    pre_p_e1 = pre_p_e1.loc[pre_p_e1['Pollo'] != 0]
    # p_e =pre_p_e1.drop_duplicates(subset = ['Pollo', 'Medida'],  keep = 'last').reset_index(drop = True)
    t    = hdf_db.get('data/tabla')
    X_train=np.zeros((pre_p_e1.shape[0],numero_muestras,numero_entradas))
    y_train=np.zeros((pre_p_e1.shape[0],1))
    x=0
    for index, row in pre_p_e1.iterrows():   # El primer registro no se toma en cuenta porque es basura
        Primero = int(row['Primero'])
        Ultimo  = int(row['Ultimo'])
        estado  = int(row['Estado'])
        #print(Primero)
        #print(Ultimo)
        #print(estado)
        if numero_clases==2:
            if estado == 0 or estado== 1:
                target = 0
            else:
                target = 1
        else:
            target=estado
        pepito=np.array(t.iloc[Primero:Ultimo+1])
        # #print(pepito.shape)
        X_train[x]=pepito[:,entrada]
        #print(X_train[x][0:4,:])       
        y_train[x]=target
        y_train_to_categorical = to_categorical(y_train)
        x=x+1



X_train_filtrado = X_train
#y_train_filtrado = y_train
y_train_filtrado = y_train_to_categorical


scaler = MinMaxScaler(feature_range=(0, 1))
#scaler = StandardScaler()



#data1=np.concatenate((X_train_filtrado,X_test_filtrado1),axis=0) 

data_2d = X_train_filtrado.reshape(-1, X_train_filtrado.shape[-1])
normalized_data_2d = scaler.fit_transform(data_2d)



X_train_Normalizado=normalized_data_2d.reshape(X_train_filtrado.shape)
y_train_Normalizado=y_train_filtrado # los valores ya estaban normalizados
print(data_2d.shape)
print(X_train_Normalizado.shape)
print(y_train_Normalizado.shape)

inputs=X_train_Normalizado.reshape(X_train_Normalizado.shape[0],-1)
targets=y_train_Normalizado

print(inputs.shape)
print(targets.shape)



(38994, 2)
(194, 201, 2)
(194, 2)
(194, 402)
(194, 2)


Vamos a hacer los conjuntos de entrenamiento validacion y test

In [13]:
factor_aprendizaje=0.001
dimension_LSTM=200
dimension_dense1=50
dimension_dense2=20
algoritmo='rmsprop'
supermax=8*4
lossfunction='categorical_crossentropy'
def create_model():

    model = Sequential()
    model.add(Bidirectional(LSTM(dimension_LSTM, return_sequences=True,recurrent_regularizer='L2',input_shape=(numero_muestras, numero_entradas))))
    model.add(Flatten())  
    #model.add(GRU(50, return_sequences=True))
    #model.add(GRU(50, return_sequences=False, recurrent_regularizer='L2'))
    model.add(Dense(dimension_dense1, activation='tanh', activity_regularizer='L2'))
    model.add(Dense(dimension_dense2, activation='tanh'))
    model.add(Dense(numero_clases, activation='softmax'))
    model.compile(loss=lossfunction, optimizer=algoritmo, metrics=['accuracy',
                              tf.keras.metrics.Recall(class_id=0),
                              tf.keras.metrics.Recall(class_id=1) #,
                              #tfa.metrics.F1Score(num_classes=numero_clases,average='macro', threshold=0.5)
                              ])
    model.optimizer.lr=(factor_aprendizaje)
    return model



In [14]:

experimento="LOMOS_Agilent_5clases_GRU1_{}_dense1_{}_dense2_{}_loss_{}_lr_{}_algoritmo_{}".format(dimension_LSTM,dimension_dense1,dimension_dense2,lossfunction,factor_aprendizaje,algoritmo)
logdir="./logs/defs/leavekout/{}_{}".format(experimento,datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback=tf.keras.callbacks.TensorBoard(log_dir=logdir, histogram_freq=1)
file_writer_cm = tf.summary.create_file_writer(logdir + '/cm')


In [15]:
if numero_clases==2:
    class_names=['Buenos', 'Malos']
else:
    class_names=['A', 'B+', 'B', 'B-','C']

In [16]:
lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.95,
    patience=1000,
    min_lr=0.0001
)
early_stop=tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', min_delta=0, patience=2000, verbose=2, mode='auto', baseline=None, restore_best_weights=True)
# Define the K-fold Cross Validator
kfold = KFold(n_splits=5, shuffle=True)
# Define per-fold score containers
acc_per_fold = []
loss_per_fold = []
sensibilidad_YAKE_per_fold=[]
sensibilidad_no_YAKE_per_fold=[]
# K-fold Cross Validation model evaluation
fold_no = 1
for train, test in kfold.split(inputs, targets):
    model=create_model()
     # Generate a print
    print('------------------------------------------------------------------------')
    print(f'Training for fold {fold_no} ...')
    inputs_good=inputs.reshape(X_train_filtrado.shape)
    # Fit data to model
    history = model.fit(inputs_good[train], targets[train],
              batch_size=20,
              epochs=numero_epochs,
              callbacks=[early_stop,lr_callback, tensorboard_callback],
              validation_data=(inputs_good[test],targets[test])
              )
    if numero_clases==2:
        target_names = ['Buenos', 'Malos']
    else:   
        target_names = ['A', 'B+', 'B', 'B-','C']
    y_pred = model.predict(inputs_good[test])
    y_pred2=np.argmax(y_pred,axis=1)
    y_test_def2=np.argmax(targets[test],axis=1)
    print(classification_report(y_test_def2, y_pred2, target_names=target_names, digits=4))
    # Generate generalization metrics
    scores = model.evaluate(inputs_good[test], targets[test], verbose=0)
    print(f'Score for fold {fold_no}: {model.metrics_names[0]} of {scores[0]}; {model.metrics_names[1]} of {scores[1]*100}%')
    print(scores[2])
    print(scores[3])
   # print(scores[4])
    acc_per_fold.append(scores[1] * 100)
    loss_per_fold.append(scores[0])
    sensibilidad_YAKE_per_fold.append(scores[2] * 100)
    sensibilidad_no_YAKE_per_fold.append(scores[3] * 100)
    # Increase fold number
    fold_no = fold_no + 1

------------------------------------------------------------------------
Training for fold 1 ...
Epoch 1/20000


/home/rgadea/experimentos_software_2024/miniconda3/envs/tensorflow_gpu_bestial/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 177ms/step - accuracy: 0.5662 - loss: 10.9685 - recall_10: 0.7986 - recall_11: 0.2305 - val_accuracy: 0.3846 - val_loss: 12.7785 - val_recall_10: 0.0000e+00 - val_recall_11: 1.0000 - learning_rate: 0.0010
Epoch 2/20000
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 116ms/step - accuracy: 0.5420 - loss: 10.0144 - recall_10: 0.5207 - recall_11: 0.5556 - val_accuracy: 0.6154 - val_loss: 4.1345 - val_recall_10: 1.0000 - val_recall_11: 0.0000e+00 - learning_rate: 0.0010
Epoch 3/20000
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 121ms/step - accuracy: 0.6087 - loss: 3.7981 - recall_10: 0.6379 - recall_11: 0.5360 - val_accuracy: 0.3590 - val_loss: 2.6789 - val_recall_10: 0.0000e+00 - val_recall_11: 0.9333 - learning_rate: 0.0010
Epoch 4/20000
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step - accuracy: 0.5236 - loss: 2.5267 - recall_10: 0.4426 - recall_11: 0.6143 - val_accuracy: 0.3590 - val_loss: 2.0838 - val_recall_10: 0.0417 - val_recall_11: 0.8667 - learning_rate: 0.0010
Epoch 5/20000
8/8 ━━━━━━━━━━━━━━━━━

KeyboardInterrupt: 

In [ ]:
# == Provide average scores ==
print('------------------------------------------------------------------------')
print('Score per fold')
for i in range(0, len(acc_per_fold)):
  print('------------------------------------------------------------------------')
  print(f'> Fold {i+1} - Loss: {loss_per_fold[i]} - Accuracy: {acc_per_fold[i]}%')
print('------------------------------------------------------------------------')
print('Average scores for all folds:')
print(f'> Accuracy: {np.mean(acc_per_fold)} (+- {np.std(acc_per_fold)})')
print(f'> Loss: {np.mean(loss_per_fold)}')
print('------------------------------------------------------------------------')

------------------------------------------------------------------------
Score per fold
------------------------------------------------------------------------
> Fold 1 - Loss: 0.7138864398002625 - Accuracy: 58.974361419677734%
------------------------------------------------------------------------
> Fold 2 - Loss: 0.6611351370811462 - Accuracy: 71.79487347602844%
------------------------------------------------------------------------
> Fold 3 - Loss: 12.258414268493652 - Accuracy: 61.538463830947876%
------------------------------------------------------------------------
> Fold 4 - Loss: 16.51393699645996 - Accuracy: 46.15384638309479%
------------------------------------------------------------------------
> Fold 5 - Loss: 15.09460163116455 - Accuracy: 57.894736528396606%
------------------------------------------------------------------------
Average scores for all folds:
> Accuracy: 59.27125632762909 (+- 8.197934048768643)
> Loss: 9.048394894599914
-----------------------------

In [ ]:
# == Provide average scores ==
print('------------------------------------------------------------------------')
print('Score per fold')
for i in range(0, len(sensibilidad_YAKE_per_fold)):
  print('------------------------------------------------------------------------')
  print(f'> Fold {i+1} - Sensibilidad YAKE: {sensibilidad_YAKE_per_fold[i]} - Sensibilidad NOYAKE: {sensibilidad_no_YAKE_per_fold[i]}%')
print('------------------------------------------------------------------------')
print('Average scores for all folds:')
print(f'> Sensibilidad YAKE: {np.mean(sensibilidad_YAKE_per_fold)} (+- {np.std(sensibilidad_YAKE_per_fold)})')
print(f'> Sensibilidad NOYAKE: {np.mean(sensibilidad_no_YAKE_per_fold)} (+- {np.std(sensibilidad_no_YAKE_per_fold)})')
print('------------------------------------------------------------------------')

------------------------------------------------------------------------
Score per fold
------------------------------------------------------------------------
> Fold 1 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 11.11111119389534%
------------------------------------------------------------------------
> Fold 2 - Sensibilidad YAKE: 90.47619104385376 - Sensibilidad NOYAKE: 50.0%
------------------------------------------------------------------------
> Fold 3 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 0.0%
------------------------------------------------------------------------
> Fold 4 - Sensibilidad YAKE: 0.0 - Sensibilidad NOYAKE: 100.0%
------------------------------------------------------------------------
> Fold 5 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 0.0%
------------------------------------------------------------------------
Average scores for all folds:
> Sensibilidad YAKE: 78.09523820877075 (+- 39.22144819197957)
> Sensibilidad NOYAKE: 32.222222238